# 03 · Random Forest

Se entrena con junio, solo en las localidades habilitadas. Se comparan tres conjuntos de variables (`variables_v2.CONJUNTOS`): `v1` (las del cuaderno 02), `v2` (v1 + tipología, más amenidades, noches mínimas recortadas y estancia mensual) y `v2_geo` (v2 + latitud y longitud).

**Esquema de evaluación**

- **Selección de hiperparámetros (secciones 0 a 3):** validación cruzada de 10 pliegues sobre junio, agrupada por anfitrión (`host_id`) y estratificada por localidad (`experimentos.pliegues_cv`). La métrica es `mdape_cv`, el MdAPE promedio de los 10 pliegues de validación.
- **Prueba final (sección 4):** el modelo elegido para cada conjunto se entrena con todo junio y se mide una sola vez en los 1.632 anuncios nuevos de septiembre (`mdape_nuevos`), que simulan el caso real de un anfitrión que publica por primera vez.



| Etapa | Sección | Experimento MLflow |
|---|---|---|
| Búsqueda preliminar: modelo por defecto y 10 configuraciones | 0 | `bogoprice-random-forest-prebusqueda-cv` |
| Un hiperparámetro a la vez desde la base | 1–2 | `bogoprice-random-forest-cv` | 
| Ajuste: combinaciones de los dos hiperparámetros más influyentes | 3 | `bogoprice-random-forest-ajuste-cv` | 
| Prueba final del modelo elegido | 4 | `bogoprice-prueba-final` | 

In [ ]:
import os
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

from bogoprice import variables_v2
from bogoprice.experimentos import (evaluar_cv, evaluar_prueba, intervalo_bootstrap,
                                    pliegues_cv, preparar_datos, resumen_pliegues)
from bogoprice.tracking import iniciar_mlflow, registrar_corrida

# Tag de autor de MLflow
AUTOR = "Edgar"

# Qué secciones se ejecutan 
EJECUTAR_PREBUSQUEDA = True    # Sección 0

datos = preparar_datos()
pliegues = pliegues_cv(datos)
resultados = []

### validación cruzada

Junio se divide en 10 partes. 

In [ ]:
display(resumen_pliegues(datos, pliegues).round(1))

## 0. Búsqueda preliminar: de los valores por defecto a la configuración base

El punto de partida es un Random Forest con los valores por defecto de scikit-learn ( 100 árboles, sin límite de profundidad, hojas de un solo anuncio , todas las variables en cada corte  y cada árbol entrenado con una muestra del tamaño completo). Se compara con 10 configuraciones, cada una pensada para probar una idea distinta sobre cómo deben crecer los árboles.

In [ ]:
if EJECUTAR_PREBUSQUEDA:
    iniciar_mlflow("bogoprice-random-forest-prebusqueda-cv")

    # Valores por defecto de scikit-learn (explícitos para que queden registrados en MLflow)
    DEFECTO = {"n_estimators": 100, "max_depth": None, "min_samples_leaf": 1,
               "max_features": 1.0, "max_samples": None}

    # (nombre, cambios respecto a defecto)
    configuraciones = [
        ("defecto",                {}),
        ("base_anterior",          {"n_estimators": 400, "min_samples_leaf": 3, "max_features": 0.5}),
        ("hoja5_mf05",             {"n_estimators": 300, "min_samples_leaf": 5, "max_features": 0.5}),
        ("hoja10_mf05",            {"n_estimators": 300, "min_samples_leaf": 10, "max_features": 0.5}),
        ("hoja5_mf03",             {"n_estimators": 300, "min_samples_leaf": 5, "max_features": 0.3}),
        ("hoja10_mf05_muestras05", {"n_estimators": 300, "min_samples_leaf": 10, "max_features": 0.5,
                                    "max_samples": 0.5}),
        ("hoja1_mf03",             {"n_estimators": 300, "max_features": 0.3}),
        ("profundidad15",          {"n_estimators": 300, "max_depth": 15}),
        ("hoja50_mf05",            {"n_estimators": 300, "min_samples_leaf": 50, "max_features": 0.5}),
        ("profundidad3",           {"n_estimators": 300, "max_depth": 3}),
        ("arboles5",               {"n_estimators": 5}),
    ]

    previa = []
    for version in ["v1", "v2", "v2_geo"]:
        for nombre_cfg, cambios in configuraciones:
            params = DEFECTO | cambios
            modelo = RandomForestRegressor(n_jobs=-1, random_state=0, **params)
            nombre = f"rf_pre_{nombre_cfg}_{version}"
            r = evaluar_cv(datos, pliegues, nombre, modelo, version,
                           {"modelo": "RandomForest", "parametro_variado": "prebusqueda", **params},
                           AUTOR)
            previa.append({"configuracion": nombre_cfg, "conjunto": version, **r})
            print(f"{version:7s} {nombre_cfg:24s} MdAPE CV = {r['mdape_cv']:.2f} ± {r['mdape_cv_std']:.2f} %  "
                  f"entrenamiento = {r['mdape_train_cv']:.2f} %")

    previa = pd.DataFrame(previa)

In [ ]:
if EJECUTAR_PREBUSQUEDA:
    # Tabla: MdAPE de validación cruzada y brecha frente a entrenamiento, por conjunto
    orden = [cfg[0] for cfg in configuraciones]
    tabla_previa = (previa.pivot_table(index="configuracion", columns="conjunto",
                                       values=["mdape_cv", "brecha_cv"])
                    .reindex(orden).round(2))
    display(tabla_previa)

    # Figura: error de entrenamiento vs. validación cruzada para cada configuración
    fig, ejes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    for eje, version in zip(ejes, ["v1", "v2", "v2_geo"]):
        d = previa[previa["conjunto"] == version].set_index("configuracion").loc[orden]
        y = range(len(d))
        eje.barh([i + 0.2 for i in y], d["mdape_train_cv"], height=0.4, label="Entrenamiento")
        eje.barh([i - 0.2 for i in y], d["mdape_cv"], height=0.4, xerr=d["mdape_cv_std"],
                 label="Validación cruzada (± desv. est.)")
        eje.axvline(20, color="red", ls="--", lw=1, label="Meta 20 %")
        eje.set_yticks(list(y), d.index)
        eje.invert_yaxis()
        eje.set_xlabel("MdAPE (%)")
        eje.set_title(f"Búsqueda preliminar — {version}")
    ejes[0].legend(loc="lower right")
    fig.tight_layout()

    # Se guarda la figura y la tabla en MLflow, en una corrida de resumen
    mejores_pre = previa.loc[previa.groupby("conjunto")["mdape_cv"].idxmin()]
    with tempfile.TemporaryDirectory() as tmp:
        ruta_fig = os.path.join(tmp, "prebusqueda_rf_cv.png")
        ruta_csv = os.path.join(tmp, "prebusqueda_rf_cv.csv")
        fig.savefig(ruta_fig, dpi=130)
        previa.to_csv(ruta_csv, index=False)
        registrar_corrida(
            "rf_pre_resumen",
            {"configuraciones": str(orden)},
            {f"mejor_mdape_cv_{fila.conjunto}": float(fila.mdape_cv) for fila in mejores_pre.itertuples()},
            artefactos=[ruta_fig, ruta_csv],
            tags={"autor": AUTOR, "modelo": "Resumen", "etapa": "validacion_cruzada"},
        )
    plt.show()